In [1]:
# Data manipulation
import pandas as pd
import numpy as np
import pickle
import json
from pathlib import Path

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix, 
                             classification_report, roc_curve)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Imbalanced data
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import warnings
warnings.filterwarnings('ignore')

# Settings
plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.max_columns', None)

print("✅ Библиотеки загружены успешно")

✅ Библиотеки загружены успешно


In [ ]:
# Загрузка данных
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f"✅ Данные загружены. Размер: {df.shape}")

NameError: name 'df' is not defined

In [4]:
# Просмотр новых признаков
print("📊 Статистика новых признаков:\n")
print(df[['total_services', 'avg_monthly_spend', 'price_per_service', 'has_internet', 'has_support']].describe())
print(f"\nTenure groups:\n{df['tenure_group'].value_counts()}")

📊 Статистика новых признаков:



NameError: name 'df' is not defined

In [ ]:
# Бинарное кодирование для Yes/No колонок
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    X[col] = X[col].map({'Yes': 1, 'No': 0})

print(f"✅ Бинарное кодирование применено к {len(binary_cols)} колонкам")

# One-Hot Encoding для остальных категориальных признаков
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"\n📋 Колонки для One-Hot Encoding: {categorical_features}")

X_encoded = pd.get_dummies(X, columns=categorical_features, drop_first=True)

print(f"\n✅ One-Hot Encoding завершен")
print(f"Размерность после кодирования: {X_encoded.shape}")
print(f"Количество признаков: {X_encoded.shape[1]}")

In [ ]:
# Сохраняем обработанные данные
X_train_scaled.to_csv('../data/processed/train.csv', index=False)
y_train_df = pd.DataFrame({'Churn': y_train})
y_train_df.to_csv('../data/processed/train_labels.csv', index=False)

X_test_scaled.to_csv('../data/processed/test.csv', index=False)
y_test_df = pd.DataFrame({'Churn': y_test})
y_test_df.to_csv('../data/processed/test_labels.csv', index=False)

print("✅ Обработанные данные сохранены в data/processed/")

In [ ]:
# Baseline: Logistic Regression
baseline_lr = LogisticRegression(random_state=42, max_iter=1000)
baseline_results = evaluate_model(baseline_lr, X_train_scaled, y_train, 
                                  X_test_scaled, y_test, "Baseline Logistic Regression")

In [ ]:
# Сравнение моделей
results_df = pd.DataFrame(results)
results_df = results_df.drop('model', axis=1)
results_df = results_df.sort_values('roc_auc', ascending=False)

print("\n📊 Сравнение моделей (отсортировано по ROC-AUC):")
print("="*80)
print(results_df.to_string(index=False))

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Сравнение метрик
metrics = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']
results_df.set_index('model_name')[metrics].plot(kind='bar', ax=axes[0], rot=45)
axes[0].set_title('Сравнение моделей по метрикам', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Score', fontsize=12)
axes[0].legend(title='Metrics', bbox_to_anchor=(1.05, 1), loc='upper left')
axes[0].grid(alpha=0.3, axis='y')

# ROC-AUC comparison
results_df.plot(x='model_name', y='roc_auc', kind='barh', ax=axes[1], 
               color='coral', edgecolor='black', alpha=0.7, legend=False)
axes[1].set_title('ROC-AUC Score по моделям', fontsize=14, fontweight='bold')
axes[1].set_xlabel('ROC-AUC Score', fontsize=12)
axes[1].set_ylabel('Model', fontsize=12)
axes[1].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('../reports/figures/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ График сохранен: reports/figures/model_comparison.png")

In [ ]:
# Сравнение с SMOTE и без
results_balanced_df = pd.DataFrame(results_balanced)
results_balanced_df = results_balanced_df.drop('model', axis=1)
results_balanced_df = results_balanced_df.sort_values('roc_auc', ascending=False)

print("\n📊 Сравнение моделей с SMOTE (отсортировано по ROC-AUC):")
print("="*80)
print(results_balanced_df.to_string(index=False))

---
## ✅ Preprocessing & Modeling Complete!

**Следующие шаги**:
- Model Evaluation & Explainability (03_model_evaluation_explainability.ipynb)
- Feature Importance Analysis
- SHAP Values
- Business Recommendations

In [ ]:
# Сохраняем финальную модель, scaler и encoder
print("\n💾 Сохранение артефактов...")

# Модель
with open('../models/churn_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print("✅ Модель сохранена: models/churn_model.pkl")

# Scaler
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✅ Scaler сохранен: models/scaler.pkl")

# Сохраняем список признаков
feature_names = X_train.columns.tolist()
with open('../models/feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)
print("✅ Feature names сохранены: models/feature_names.pkl")

# Сохраняем метрики
metrics_dict = {
    'best_model': best_model_name,
    'roc_auc': best_model_info['roc_auc'],
    'accuracy': best_model_info['accuracy'],
    'precision': best_model_info['precision'],
    'recall': best_model_info['recall'],
    'f1_score': best_model_info['f1_score']
}

with open('../reports/metrics.json', 'w') as f:
    json.dump(metrics_dict, f, indent=4)
print("✅ Метрики сохранены: reports/metrics.json")

print("\n🎉 Все артефакты сохранены успешно!")

In [ ]:
# Выбираем лучшую модель (по ROC-AUC)
best_model_info = results_balanced[0]  # Уже отсортировано
best_model = best_model_info['model']
best_model_name = best_model_info['model_name']

print(f"🏆 Лучшая модель: {best_model_name}")
print(f"   ROC-AUC: {best_model_info['roc_auc']:.4f}")
print(f"   F1-Score: {best_model_info['f1_score']:.4f}")

## 9. Final Model Selection & Saving

In [ ]:
# Переобучаем лучшую модель на сбалансированных данных
print("\n🚀 Обучение моделей на сбалансированных данных...\n")

results_balanced = []
for name, model in models.items():
    result = evaluate_model(model, X_train_balanced, y_train_balanced, 
                           X_test_scaled, y_test, f"{name} + SMOTE")
    results_balanced.append(result)

print(f"\n{'='*60}")
print("✅ Модели с SMOTE обучены!")

In [ ]:
# Применяем SMOTE для балансировки классов
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print(f"📊 Размер до SMOTE: {X_train_scaled.shape}")
print(f"📊 Размер после SMOTE: {X_train_balanced.shape}")
print(f"\n🎯 Распределение до SMOTE:")
print(y_train.value_counts())
print(f"\n🎯 Распределение после SMOTE:")
print(pd.Series(y_train_balanced).value_counts())

## 8. Handling Class Imbalance with SMOTE

In [ ]:
# Тестируем несколько моделей
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=10),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100, max_depth=15),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=100, max_depth=5),
    'XGBoost': XGBClassifier(random_state=42, n_estimators=100, max_depth=5, eval_metric='logloss')
}

results = []

print("🚀 Обучение и тестирование моделей...\n")
for name, model in models.items():
    result = evaluate_model(model, X_train_scaled, y_train, X_test_scaled, y_test, name)
    results.append(result)

print(f"\n{'='*60}")
print("✅ Все модели обучены!")

## 7. Model Comparison (Multiple Models)

In [ ]:
# Функция для оценки модели
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name="Model"):
    """Обучает и оценивает модель"""
    
    # Обучение
    model.fit(X_train, y_train)
    
    # Предсказания
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Метрики
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    # Вывод результатов
    print(f"\n{'='*60}")
    print(f"🎯 {model_name} - Результаты на тестовой выборке")
    print(f"{'='*60}")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")
    
    return {
        'model_name': model_name,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'roc_auc': roc_auc,
        'model': model
    }

print("✅ Функция evaluate_model создана")

## 6. Baseline Model (Logistic Regression)

In [ ]:
# Стандартизация признаков
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Конвертируем обратно в DataFrame для удобства
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print(f"✅ Признаки стандартизированы")
print(f"Mean после scaling (должно быть ≈0): {X_train_scaled.mean().mean():.6f}")
print(f"Std после scaling (должно быть ≈1): {X_train_scaled.std().mean():.6f}")

In [ ]:
# Разделение на train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print(f"📊 Разделение данных:")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"\n🎯 Распределение в train:")
print(y_train.value_counts(normalize=True))
print(f"\n🎯 Распределение в test:")
print(y_test.value_counts(normalize=True))

## 5. Train-Test Split & Scaling

In [ ]:
# Определяем типы признаков
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"📊 Числовых признаков: {len(numeric_features)}")
print(f"📋 Категориальных признаков: {len(categorical_features)}")
print(f"\n📊 Числовые: {numeric_features}")
print(f"\n📋 Категориальные: {categorical_features}")

In [ ]:
# Разделяем на признаки и целевую переменную
X = df.drop('Churn', axis=1)
y = df['Churn'].map({'No': 0, 'Yes': 1})

print(f"📊 Размерность:")
print(f"X: {X.shape}")
print(f"y: {y.shape}")
print(f"\nЦелевая переменная:\n{y.value_counts()}")

## 4. Encoding & Data Preparation

In [ ]:
# Создание новых признаков
print("🔨 Feature Engineering...")

# 1. Tenure группировка
df['tenure_group'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72], 
                             labels=['0-1 year', '1-2 years', '2-4 years', '4+ years'])

# 2. Средняя стоимость за месяц от общей
df['avg_monthly_spend'] = df['TotalCharges'] / (df['tenure'] + 1)  # +1 чтобы избежать деления на 0

# 3. Количество услуг
services = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
           'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

df['total_services'] = 0
for col in services:
    df['total_services'] += (df[col] == 'Yes').astype(int)

# 4. Имеет ли интернет
df['has_internet'] = (df['InternetService'] != 'No').astype(int)

# 5. Имеет ли доп услуги поддержки
df['has_support'] = ((df['OnlineSecurity'] == 'Yes') | 
                     (df['TechSupport'] == 'Yes') |
                     (df['DeviceProtection'] == 'Yes')).astype(int)

# 6. Соотношение MonthlyCharges к количеству услуг
df['price_per_service'] = df['MonthlyCharges'] / (df['total_services'] + 1)

print(f"✅ Создано {6} новых признаков")
print(f"Новые колонки: tenure_group, avg_monthly_spend, total_services, has_internet, has_support, price_per_service")
print(f"\nРазмер датасета: {df.shape}")

## 3. Feature Engineering

In [ ]:
# Очистка TotalCharges
print("🔧 Очистка TotalCharges...")

# Находим пустые значения (пробелы)
whitespace_mask = df['TotalCharges'].str.strip() == ''
print(f"Найдено пустых значений: {whitespace_mask.sum()}")

# Конвертируем в float
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Заполняем NaN значениями MonthlyCharges (так как tenure=0)
df['TotalCharges'].fillna(df['MonthlyCharges'], inplace=True)

print(f"✅ TotalCharges очищен. Тип данных: {df['TotalCharges'].dtype}")
print(f"   Пропусков: {df['TotalCharges'].isnull().sum()}")

In [ ]:
# Загрузка данных
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f"📦 Исходный датасет: {df.shape}")
print(f"Target distribution:\n{df['Churn'].value_counts()}")

## 2. Загрузка и очистка данных

## 1. Импорт библиотек

# 🛠️ Data Preprocessing & Model Training

**Цель**: Подготовка данных, создание признаков и обучение моделей для прогнозирования оттока

---

## Оглавление
1. Загрузка и очистка данных
2. Feature Engineering
3. Data Splitting & Preprocessing Pipeline
4. Baseline Model
5. Model Selection & Hyperparameter Tuning
6. Final Model Training
7. Сохранение моделей и артефактов